## 2.1 Fixed-Size Chunking

A estratégia mais simples: divida o texto a cada N caracteres, com overlap de M caracteres.

**Vantagem:** previsível, fácil de implementar, sem dependências.

**Desvantagem:** ignora completamente a estrutura do texto. Pode cortar no meio de:
- Uma frase: "O algoritmo HNSW usa uma estrutura / hierárquica para..."
- Um parágrafo: informação logicamente conectada fica em chunks separados
- Um código: função cortada ao meio não tem sentido isolada

**Quando usar:** protótipos, textos muito uniformes (transcrições de fala, logs), quando velocidade de implementação é prioridade.

**Parâmetros típicos:** chunk_size=500 chars, overlap=50 chars (10% do tamanho).

In [ ]:
## 2.2 Recursive Character Split

Uma evolução do fixed-size: em vez de cortar a cada N caracteres, tenta respeitar a hierarquia natural do texto.

**Como funciona:** define uma lista de separadores em ordem de preferência:
1. `

` (parágrafo) — divide aqui se possível
2. `
` (linha) — se o chunk ainda for grande, divide por linha
3. `. ` (frase) — se ainda for grande, divide por frase
4. ` ` (palavra) — último recurso
5. `""` (caractere) — último, último recurso

O algoritmo aplica esses separadores recursivamente até que todos os chunks estejam abaixo do tamanho máximo.

**Resultado:** chunks que respeitam a estrutura natural do documento, sem quebrar parágrafos ou frases no meio.

**Este é o padrão recomendado para a maioria dos casos.** LangChain e LlamaIndex usam isso como default.

## 2.1 Fixed-Size Chunking

A estratégia mais simples: divida o texto a cada N caracteres, com overlap de M caracteres.

**Vantagem:** previsível, fácil de implementar, sem dependências.

**Desvantagem:** ignora completamente a estrutura do texto. Pode cortar no meio de:
- Uma frase: "O algoritmo HNSW usa uma estrutura / hierárquica para..."
- Um parágrafo: informação logicamente conectada fica em chunks separados
- Um código: função cortada ao meio não tem sentido isolada

**Quando usar:** protótipos, textos muito uniformes (transcrições de fala, logs), quando velocidade de implementação é prioridade.

**Parâmetros típicos:** chunk_size=500 chars, overlap=50 chars (10% do tamanho).

In [ ]:
chunks_fixed = fixed_size_chunk(texto_exemplo, chunk_size=500, overlap=50)

print(f'Fixed-size (500 chars, 50 overlap):')
print(f'  Numero de chunks: {len(chunks_fixed)}')
print(f'  Chars medios: {sum(len(c.text) for c in chunks_fixed)/len(chunks_fixed):.0f}')
print(f'  Tokens medios: {sum(c.token_count for c in chunks_fixed)/len(chunks_fixed):.0f}')
print(f'\nPrimeiro chunk:')
print(repr(chunks_fixed[0].text[:200]))
print(f'\nSegundo chunk (mostrando overlap):')
print(repr(chunks_fixed[1].text[:200]))

## 2.2 Recursive Character Split

Uma evolução do fixed-size: em vez de cortar a cada N caracteres, tenta respeitar a hierarquia natural do texto.

**Como funciona:** define uma lista de separadores em ordem de preferência:
1. `

` (parágrafo) — divide aqui se possível
2. `
` (linha) — se o chunk ainda for grande, divide por linha
3. `. ` (frase) — se ainda for grande, divide por frase
4. ` ` (palavra) — último recurso
5. `""` (caractere) — último, último recurso

O algoritmo aplica esses separadores recursivamente até que todos os chunks estejam abaixo do tamanho máximo.

**Resultado:** chunks que respeitam a estrutura natural do documento, sem quebrar parágrafos ou frases no meio.

**Este é o padrão recomendado para a maioria dos casos.** LangChain e LlamaIndex usam isso como default.

In [ ]:
## 2.2 Recursive Character Split

Uma evolução do fixed-size: em vez de cortar a cada N caracteres, tenta respeitar a hierarquia natural do texto.

**Como funciona:** define uma lista de separadores em ordem de preferência:
1. `

` (parágrafo) — divide aqui se possível
2. `
` (linha) — se o chunk ainda for grande, divide por linha
3. `. ` (frase) — se ainda for grande, divide por frase
4. ` ` (palavra) — último recurso
5. `""` (caractere) — último, último recurso

O algoritmo aplica esses separadores recursivamente até que todos os chunks estejam abaixo do tamanho máximo.

**Resultado:** chunks que respeitam a estrutura natural do documento, sem quebrar parágrafos ou frases no meio.

**Este é o padrão recomendado para a maioria dos casos.** LangChain e LlamaIndex usam isso como default.

## 2.3 Semantic Chunking

A estratégia mais sofisticada: divide onde o *significado* muda, não onde o texto tem quebras.

**Como funciona:**
1. Divide o texto em sentenças
2. Computa o embedding de cada sentença
3. Calcula a similaridade entre sentenças consecutivas
4. Quando a similaridade cai abruptamente (mudança de tópico), cria um novo chunk

**Exemplo:** num artigo que discute Python e depois machine learning, a fronteira semântica aparecerá quando o texto muda de "sintaxe de listas" para "gradient descent" — mesmo que não haja um parágrafo explícito separando.

**Custo:** caro. Precisa embedar todas as sentenças antes de chunkar. Para 1000 documentos, isso multiplica o tempo de indexação por 3-5x.

**Quando vale a pena:** documentos longos e heterogêneos (livros, relatórios extensos, transcrições de reuniões longas) onde mudanças de tópico são frequentes e importantes.

In [ ]:
## 2.3 Semantic Chunking

A estratégia mais sofisticada: divide onde o *significado* muda, não onde o texto tem quebras.

**Como funciona:**
1. Divide o texto em sentenças
2. Computa o embedding de cada sentença
3. Calcula a similaridade entre sentenças consecutivas
4. Quando a similaridade cai abruptamente (mudança de tópico), cria um novo chunk

**Exemplo:** num artigo que discute Python e depois machine learning, a fronteira semântica aparecerá quando o texto muda de "sintaxe de listas" para "gradient descent" — mesmo que não haja um parágrafo explícito separando.

**Custo:** caro. Precisa embedar todas as sentenças antes de chunkar. Para 1000 documentos, isso multiplica o tempo de indexação por 3-5x.

**Quando vale a pena:** documentos longos e heterogêneos (livros, relatórios extensos, transcrições de reuniões longas) onde mudanças de tópico são frequentes e importantes.

## 3.4 Comparacao: Impacto na Qualidade de Retrieval

In [ ]:
## 2.2 Recursive Character Split

Uma evolução do fixed-size: em vez de cortar a cada N caracteres, tenta respeitar a hierarquia natural do texto.

**Como funciona:** define uma lista de separadores em ordem de preferência:
1. `

` (parágrafo) — divide aqui se possível
2. `
` (linha) — se o chunk ainda for grande, divide por linha
3. `. ` (frase) — se ainda for grande, divide por frase
4. ` ` (palavra) — último recurso
5. `""` (caractere) — último, último recurso

O algoritmo aplica esses separadores recursivamente até que todos os chunks estejam abaixo do tamanho máximo.

**Resultado:** chunks que respeitam a estrutura natural do documento, sem quebrar parágrafos ou frases no meio.

**Este é o padrão recomendado para a maioria dos casos.** LangChain e LlamaIndex usam isso como default.

In [ ]:
## 2.2 Recursive Character Split

Uma evolução do fixed-size: em vez de cortar a cada N caracteres, tenta respeitar a hierarquia natural do texto.

**Como funciona:** define uma lista de separadores em ordem de preferência:
1. `

` (parágrafo) — divide aqui se possível
2. `
` (linha) — se o chunk ainda for grande, divide por linha
3. `. ` (frase) — se ainda for grande, divide por frase
4. ` ` (palavra) — último recurso
5. `""` (caractere) — último, último recurso

O algoritmo aplica esses separadores recursivamente até que todos os chunks estejam abaixo do tamanho máximo.

**Resultado:** chunks que respeitam a estrutura natural do documento, sem quebrar parágrafos ou frases no meio.

**Este é o padrão recomendado para a maioria dos casos.** LangChain e LlamaIndex usam isso como default.

## 2.2 Recursive Character Split

Uma evolução do fixed-size: em vez de cortar a cada N caracteres, tenta respeitar a hierarquia natural do texto.

**Como funciona:** define uma lista de separadores em ordem de preferência:
1. `

` (parágrafo) — divide aqui se possível
2. `
` (linha) — se o chunk ainda for grande, divide por linha
3. `. ` (frase) — se ainda for grande, divide por frase
4. ` ` (palavra) — último recurso
5. `""` (caractere) — último, último recurso

O algoritmo aplica esses separadores recursivamente até que todos os chunks estejam abaixo do tamanho máximo.

**Resultado:** chunks que respeitam a estrutura natural do documento, sem quebrar parágrafos ou frases no meio.

**Este é o padrão recomendado para a maioria dos casos.** LangChain e LlamaIndex usam isso como default.

## Resumo

| Estratégia | Recall típico | Custo | Recomendação |
|-----------|--------------|-------|--------------|
| Fixed-size | Baixo-médio | O(1) | Só para protótipos |
| Recursive split | Médio-alto | O(N) | **Padrão recomendado** |
| Semantic | Alto | O(N×M) | Documentos longos e heterogêneos |

**Parâmetros a experimentar:**
- `chunk_size`: 256 (respostas curtas e precisas) a 1024 (contexto rico)
- `overlap`: 10-20% do chunk_size
- Para código: chunke por função/classe, não por tamanho

**Próximos passos:**
- [03 — Retrieval Strategies](03_retrieval_strategies.html): uma vez que você tem bons chunks, como encontrá-los eficientemente?